# Modelagem — Treinamento e Otimização de Modelos

Este notebook cobre o processo completo de modelagem para previsão de preços de computadores:

1. **Modelos lineares** — baseline e otimização de hiperparâmetros (Ridge, Lasso, ElasticNet)
2. **Modelos baseados em árvore** — baseline (Random Forest, XGBoost, LightGBM, CatBoost)
3. **Otimização via RandomizedSearchCV** — busca dos melhores hiperparâmetros
4. **Avaliação final** — validação cruzada dos modelos otimizados

## 1. Importação de Bibliotecas

Carregamento das bibliotecas necessárias para modelagem e otimização.

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_squared_error
import optuna
from sklearn.model_selection import cross_val_score
from utils import avaliar_modelo

## 2. Carregamento dos Dados

Leitura dos datasets previamente tratados:
- `linear.csv` — features preparadas para modelos lineares
- `tree.csv` — features preparadas para modelos baseados em árvore

In [2]:
df_linear = pd.read_csv(os.path.join(os.getcwd(), 'csv_gerados', 'linear.csv'))

df_tree = pd.read_csv(os.path.join(os.getcwd(), 'csv_gerados', 'tree.csv'))

k_fold = 5

X_lin = df_linear.drop(columns=['price_log'])
y_lin = df_linear['price_log']

X_tree = df_tree.drop(columns=['price_log'])
y_tree = df_tree['price_log']

## 3. Modelos Lineares

Avaliação dos modelos lineares como baseline. A variável-alvo é `price_log` (preço com transformação logarítmica).
São testados: **Linear Regression**, **Ridge**, **Lasso** e **ElasticNet** com validação cruzada K-Fold (k=5).

### 3.1 Baseline — Modelos Lineares sem Regularização

In [29]:
modelos_lineares = { 
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(random_state=42),
    "Lasso": Lasso(random_state = 42),
    "ElasticNet": ElasticNet(random_state=42)
}

for modelo in modelos_lineares:
    avaliar_modelo(modelos_lineares[modelo], X_lin,y_lin, modelo,cv = k_fold, transformacao = "log1p")


 LINEAR REGRESSION (K-Fold CV = 5)
--------------------------------------------------------------------------------------------
  Métrica  | Treino      | Validação   | Gap (V-T)  | IC 95% (Validação)      | Margem (±)
  -------- | ----------- | ----------- | ---------- | ----------------------- | ----------
  MAE      | $  207.49 | $  207.52 | +$   0.03  | [$  205.63, $  209.40] | ±$   1.88
  RMSE     | $  263.85 | $  263.88 | +$   0.03  | [$  261.14, $  266.61] | ±$   2.74
  R²       |    0.7812 |    0.7811 |   -0.0001  | [   0.7760,    0.7862] |  ± 0.0051


 RIDGE (K-Fold CV = 5)
--------------------------------------------------------------------------------------------
  Métrica  | Treino      | Validação   | Gap (V-T)  | IC 95% (Validação)      | Margem (±)
  -------- | ----------- | ----------- | ---------- | ----------------------- | ----------
  MAE      | $  207.48 | $  207.51 | +$   0.03  | [$  205.63, $  209.40] | ±$   1.88
  RMSE     | $  263.85 | $  263.88 | +$   0.03  |

> ### Analise Linear Inicial
> 
> **Linear e Ridge** com desempenho bem semelhante e R² de 0.78
>
> Falta buscar melhores hiperparametros pro ElasticNet e Lasso

### 3.2 Otimização de Hiperparâmetros — RidgeCV, LassoCV, ElasticNetCV

Busca do melhor coeficiente de regularização (alpha) via validação cruzada.
- **Ridge**: 100 valores entre 0.001 e 1000 (escala logarítmica)
- **Lasso**: 100 valores entre 0.00001 e 1.0
- **ElasticNet**: mesmos alphas do Lasso + 9 valores de l1_ratio

In [ ]:
# Gerando várias opções na escala logaritmica
alphas_ridge = np.logspace(-3, 3, 100)  # 100 chutes entre 0.001 e 1000
alphas_lasso = np.logspace(-5, 0, 100)  # 100 chutes entre 0.00001 e 1.0
l1_ratios = [0.0001, 0.001, 0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99]

# A busca descobre qual minimiza o erro
busca_ridge = RidgeCV(alphas=alphas_ridge, cv=k_fold).fit(X_lin, y_lin)
busca_lasso = LassoCV(alphas=alphas_lasso, cv=k_fold, max_iter=10000, random_state=42).fit(X_lin, y_lin)
busca_elastic = ElasticNetCV(alphas=alphas_lasso, l1_ratio=l1_ratios, cv=k_fold, max_iter=10000, random_state=42).fit(X_lin, y_lin)

print(f"Melhor Alpha Ridge:      {busca_ridge.alpha_:.5f}")
print(f"Melhor Alpha Lasso:      {busca_lasso.alpha_:.5f}")
print(f"Melhor Alpha ElasticNet: {busca_elastic.alpha_:.5f} | l1_ratio: {busca_elastic.l1_ratio_}\n")

Melhor Alpha Ridge:      0.02477
Melhor Alpha Lasso:      0.00001
Melhor Alpha ElasticNet: 0.00001 | l1_ratio: 0.0001



### 3.3 Melhores Hiperparâmetros Encontrados (Modelos Lineares)

| Modelo | Melhor Alpha | l1_ratio |
|--------|-------------|----------|
| Ridge | 0.02477 | — |
| Lasso | 0.00001 | — |
| ElasticNet | 0.00001 | 0.0001 |

>
>
> Os coeficientes de penalidade encontrados foram extremamente baixos, indicando que a regularização teve impacto mínimo.
>

### 3.4 Avaliação dos Modelos Lineares Otimizados

Reavaliação com os melhores hiperparâmetros encontrados na etapa anterior.

In [30]:
modelos_lineares_otimizados = { 
    "Ridge Otimizado": Ridge(alpha=busca_ridge.alpha_, random_state=42),
    "Lasso Otimizado": Lasso(alpha=busca_lasso.alpha_, max_iter=10000, random_state=42),
    "ElasticNet Otimizado": ElasticNet(
        alpha=busca_elastic.alpha_, 
        l1_ratio=busca_elastic.l1_ratio_, 
        max_iter=10000, 
        random_state=42
    )
}

for nome, modelo in modelos_lineares_otimizados.items():
    avaliar_modelo(modelo, X_lin, y_lin, nome, cv=k_fold, transformacao="log1p")


 RIDGE OTIMIZADO (K-Fold CV = 5)
--------------------------------------------------------------------------------------------
  Métrica  | Treino      | Validação   | Gap (V-T)  | IC 95% (Validação)      | Margem (±)
  -------- | ----------- | ----------- | ---------- | ----------------------- | ----------
  MAE      | $  207.49 | $  207.52 | +$   0.03  | [$  205.63, $  209.40] | ±$   1.88
  RMSE     | $  263.85 | $  263.88 | +$   0.03  | [$  261.14, $  266.61] | ±$   2.74
  R²       |    0.7812 |    0.7811 |   -0.0001  | [   0.7760,    0.7862] |  ± 0.0051


 LASSO OTIMIZADO (K-Fold CV = 5)
--------------------------------------------------------------------------------------------
  Métrica  | Treino      | Validação   | Gap (V-T)  | IC 95% (Validação)      | Margem (±)
  -------- | ----------- | ----------- | ---------- | ----------------------- | ----------
  MAE      | $  207.48 | $  207.51 | +$   0.03  | [$  205.63, $  209.39] | ±$   1.88
  RMSE     | $  263.84 | $  263.87 | +$  

> ### Modelos Lineares Otimizados
> 
> **Todos os modelos são consistentes** apresentam bom desempenho (R² = 0,78) e pelo baixo gap entre treino e validação não mostram apresentar graves overfitting e underfitting
>
> Modelo Lasso apresentou 1 centavo de vantagem no contexto de RMSE, vamos tomar o modelo linear como **referência mínima** para avaliar a qualidade/vantagem dos modelos de árvore **que tendem a gerar melhores resultados por óbvio**

## 4. Modelos Baseados em Árvore

Modelos não-lineares aplicados sobre o dataset `tree.csv`, que contém features adequadas para esse tipo de algoritmo.

### 4.1 Baseline — Modelos sem Otimização

Avaliação com configurações padrão (100 estimadores) para estabelecer uma linha de base de desempenho.

In [32]:
# ====================================================================
# CONFIGURAÇÃO DOS MODELOS NÃO-LINEARES (MODELOS BASELINE)
# ====================================================================
modelos_nao_lineares = {
    
    "Random Forest": RandomForestRegressor(
        n_estimators=100,      
        random_state=42,
        n_jobs=-1
    ),
    
    "XGBoost": XGBRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    
    "LightGBM": LGBMRegressor(
        n_estimators=100,
        random_state=42,
        verbose=-1,
        n_jobs=1
    ),
    
    "CatBoost": CatBoostRegressor(
        iterations=100,        
        random_seed=42,
        verbose=False
    )
}

# ====================================================================
# AVALIAÇÃO VIA VALIDAÇÃO CRUZADA
# ====================================================================
print(">>> AVALIANDO MODELOS DE ÁRVORE (BASELINE) <<<\n")

k_fold = 5  # Garante que a variável existe

for nome, modelo in modelos_nao_lineares.items():
    avaliar_modelo(
        modelo=modelo, 
        X=X_tree, 
        y=y_tree, 
        nome_modelo=nome, 
        cv=k_fold, 
        transformacao="log1p"
    )

>>> AVALIANDO MODELOS DE ÁRVORE (BASELINE) <<<


 RANDOM FOREST (K-Fold CV = 5)
--------------------------------------------------------------------------------------------
  Métrica  | Treino      | Validação   | Gap (V-T)  | IC 95% (Validação)      | Margem (±)
  -------- | ----------- | ----------- | ---------- | ----------------------- | ----------
  MAE      | $   56.73 | $  152.03 | +$  95.30  | [$  149.61, $  154.46] | ±$   2.42
  RMSE     | $   74.42 | $  197.15 | +$ 122.73  | [$  194.11, $  200.19] | ±$   3.04
  R²       |    0.9826 |    0.8778 |   -0.1048  | [   0.8749,    0.8808] |  ± 0.0030


 XGBOOST (K-Fold CV = 5)
--------------------------------------------------------------------------------------------
  Métrica  | Treino      | Validação   | Gap (V-T)  | IC 95% (Validação)      | Margem (±)
  -------- | ----------- | ----------- | ---------- | ----------------------- | ----------
  MAE      | $  121.04 | $  139.09 | +$  18.05  | [$  137.15, $  141.03] | ±$   1.94
  R

> ### Modelos de Árvore iniciais ( Baseline )
> 
> **Random Forest (Bagging)** apresenta alto overfitting, dado pelo R² de 0.98 (bizarramente alto) e pelo gap absurdo entre treino e validação > 10%. Descartaremos nas análises futuras
>
> **XGBoost** apresenta um overfitting moderado, gap(V-T) de 23$ no RMSE na faixa de 11% da validação
>
> **LightGBM e Catboost**, mesmo sem muitos parâmetros, já apresentaram resultados bem consistentes. Gaps baixos entre treino e validação e resultados na faixa de R² = 0.9

### 4.2 Preparação para Otimização — Scorer Customizado

Criação de uma métrica de avaliação que calcula o **RMSE em dólares** (revertendo a transformação logarítmica),
usada como critério de otimização no RandomizedSearchCV.

In [33]:
def rmse_em_dolares(y_true_log, y_pred_log):
    y_true_real = np.expm1(y_true_log)
    y_pred_real = np.expm1(y_pred_log)
    return np.sqrt(mean_squared_error(y_true_real, y_pred_real))

meu_scorer = make_scorer(rmse_em_dolares, greater_is_better=False)

### 4.3 Otimização via Optuna

Busca bayesiana (TPE) com **50 trials** por modelo e validação cruzada (K-Fold, k=5).
O critério de otimização é o **RMSE em dólares** (scorer customizado definido acima), revertendo a transformação `log1p` para avaliar o erro na escala original.

#### XGBoost

Espaço de busca contínua (Optuna):

| Parâmetro | Intervalo | Escala |
|-----------|-----------|--------|
| `n_estimators` | 300 – 1000 | inteira |
| `learning_rate` | 0.01 – 0.1 | logarítmica |
| `max_depth` | 4 – 10 | inteira |
| `subsample` | 0.5 – 1.0 | contínua |
| `colsample_bytree` | 0.5 – 1.0 | contínua |
| `reg_alpha` | 0.0 – 5.0 | contínua |
| `reg_lambda` | 1.0 – 10.0 | contínua |

In [ ]:
print(">>> OTIMIZANDO XGBOOST <<<\n")

def objective_xgb(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0)
    }
    
    modelo = XGBRegressor(**param, random_state=42, n_jobs=-1)
    score = cross_val_score(modelo, X_tree, y_tree, scoring=meu_scorer, cv=k_fold, n_jobs=-1)
    return -score.mean()

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=50)

print(f"\nMelhor RMSE estimado (XGBoost): ${study_xgb.best_value:.2f}")
print("Melhores Parâmetros:", study_xgb.best_params)

**Melhores hiperparâmetros encontrados (XGBoost) — Trial 49/50:**

| Parâmetro | Valor |
|-----------|-------|
| `n_estimators` | 953 |
| `learning_rate` | 0.0313 |
| `max_depth` | 4 |
| `subsample` | 0.592 |
| `colsample_bytree` | 0.812 |
| `reg_alpha` | 1.693 |
| `reg_lambda` | 4.643 |

> Melhor RMSE estimado (CV): **$173.86**

#### LightGBM

Espaço de busca contínua (Optuna):

| Parâmetro | Intervalo | Escala |
|-----------|-----------|--------|
| `n_estimators` | 300 – 1000 | inteira |
| `learning_rate` | 0.01 – 0.1 | logarítmica |
| `max_depth` | 4 – 12 | inteira |
| `num_leaves` | 20 – 100 | inteira |
| `subsample` | 0.5 – 1.0 | contínua |
| `colsample_bytree` | 0.5 – 1.0 | contínua |

> `subsample_freq` = 1 (fixo, necessário para ativar o `subsample` no LightGBM)

In [ ]:
print(">>> OTIMIZANDO LIGHTGBM <<<\n")

def objective_lgbm(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }
    
    modelo = LGBMRegressor(**param, random_state=42, verbose=-1, n_jobs=-1)
    score = cross_val_score(modelo, X_tree, y_tree, scoring=meu_scorer, cv=k_fold, n_jobs=1)
    return -score.mean()

study_lgbm = optuna.create_study(direction='minimize')
study_lgbm.optimize(objective_lgbm, n_trials=50)

print(f"\nMelhor RMSE estimado (LightGBM): ${study_lgbm.best_value:.2f}")
print("Melhores Parâmetros:", study_lgbm.best_params)

**Melhores hiperparâmetros encontrados (LightGBM) — Trial 46/50:**

| Parâmetro | Valor |
|-----------|-------|
| `n_estimators` | 959 |
| `learning_rate` | 0.0437 |
| `max_depth` | 4 |
| `num_leaves` | 50 |
| `subsample` | 0.659 |
| `subsample_freq` | 1 (fixo) |
| `colsample_bytree` | 0.596 |

> Melhor RMSE estimado (CV): **$173.94**

#### CatBoost

Espaço de busca contínua (Optuna):

| Parâmetro | Intervalo | Escala |
|-----------|-----------|--------|
| `iterations` | 300 – 1200 | inteira |
| `learning_rate` | 0.01 – 0.12 | logarítmica |
| `depth` | 3 – 8 | inteira |
| `l2_leaf_reg` | 1.0 – 10.0 | contínua |
| `subsample` | 0.5 – 1.0 | contínua |

In [ ]:
print(">>> OTIMIZANDO CATBOOST <<<\n")

def objective_cat(trial):
    param = {
        'iterations': trial.suggest_int('iterations', 300, 1200),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.12, log=True),
        'depth': trial.suggest_int('depth', 3, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0)
    }
    
    modelo = CatBoostRegressor(**param, random_seed=42, verbose=False, thread_count=-1)
    score = cross_val_score(modelo, X_tree, y_tree, scoring=meu_scorer, cv=k_fold, n_jobs=1)
    return -score.mean()

study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(objective_cat, n_trials=50)

print(f"\nMelhor RMSE estimado (CatBoost): ${study_cat.best_value:.2f}")
print("Melhores Parâmetros:", study_cat.best_params)

**Melhores hiperparâmetros encontrados (CatBoost) — Trial 16/50:**

| Parâmetro | Valor |
|-----------|-------|
| `iterations` | 849 |
| `learning_rate` | 0.0593 |
| `depth` | 5 |
| `l2_leaf_reg` | 9.819 |
| `subsample` | 0.720 |

> Melhor RMSE estimado (CV): **$173.22** — melhor resultado entre os três modelos.

## 5. Avaliação Final dos Modelos Otimizados

Validação cruzada (K-Fold, k=5) dos três modelos com os melhores hiperparâmetros encontrados pelo Optuna.
As métricas são reportadas em dólares após reverter a transformação logarítmica (`expm1`).

> **Atalho:** a célula abaixo instancia os modelos diretamente com os hiperparâmetros otimizados.
>
> permitindo rodar a avaliação final **sem precisar re-executar toda a busca do Optuna**.

In [34]:
# ====================================================================
# MODELOS OTIMIZADOS — Hiperparâmetros via Optuna 
# ====================================================================

k_fold = 5
X_tree = df_tree.drop(columns=['price_log'])
y_tree = df_tree['price_log']

modelos_otimizados = {

    "XGBoost Otimizado": XGBRegressor(
        n_estimators=953,
        learning_rate=0.0313,
        max_depth=4,
        subsample=0.592,
        colsample_bytree=0.812,
        reg_alpha=1.693,
        reg_lambda=4.643,
        random_state=42,
        n_jobs=-1
    ),

    "LightGBM Otimizado": LGBMRegressor(
        n_estimators=959,
        learning_rate=0.0437,
        max_depth=4,
        num_leaves=50,
        subsample=0.659,
        subsample_freq=1,
        colsample_bytree=0.596,
        random_state=42,
        verbose=-1,
        n_jobs=1
    ),

    "CatBoost Otimizado": CatBoostRegressor(
        iterations=849,
        learning_rate=0.0593,
        depth=5,
        l2_leaf_reg=9.819,
        subsample=0.720,
        bootstrap_type='Bernoulli',
        random_seed=42,
        verbose=False,
        thread_count=-1
    )
}

In [35]:
print(">>> AVALIAÇÃO FINAL CRUZADA DOS MODELOS OTIMIZADOS <<<\n")

for nome, modelo in modelos_otimizados.items():
    avaliar_modelo(
        modelo=modelo,
        X=X_tree,
        y=y_tree,
        nome_modelo=nome,
        cv=k_fold,
        transformacao="log1p"
    )

>>> AVALIAÇÃO FINAL CRUZADA DOS MODELOS OTIMIZADOS <<<


 XGBOOST OTIMIZADO (K-Fold CV = 5)
--------------------------------------------------------------------------------------------
  Métrica  | Treino      | Validação   | Gap (V-T)  | IC 95% (Validação)      | Margem (±)
  -------- | ----------- | ----------- | ---------- | ----------------------- | ----------
  MAE      | $  131.01 | $  134.13 | +$   3.12  | [$  132.65, $  135.61] | ±$   1.48
  RMSE     | $  169.81 | $  173.92 | +$   4.11  | [$  171.65, $  176.18] | ±$   2.27
  R²       |    0.9094 |    0.9049 |   -0.0044  | [   0.9031,    0.9067] |  ± 0.0018


 LIGHTGBM OTIMIZADO (K-Fold CV = 5)
--------------------------------------------------------------------------------------------
  Métrica  | Treino      | Validação   | Gap (V-T)  | IC 95% (Validação)      | Margem (±)
  -------- | ----------- | ----------- | ---------- | ----------------------- | ----------
  MAE      | $  128.91 | $  134.12 | +$   5.21  | [$  132.54, $  

> ### Modelos de Árvore Otimizados
> 
> **Catboost apresentou um melhor desempenho geral**, muito embora o LightGBM apresentou um bom desempenho também, com baixas diferenças entre valores de RMSE/MAE/R² 
>
> Gap RMSE V-T de apenas 4.31$, 2,5% em relação a validação.
>
> Intervalos de confiança com margens bem pequenas ( apenas 2.35$ do RMSE )
